# **Asistente Experto en Nutrición y Entrenamiento Deportivo**
## **Proyecto Final - IA Generativa con Gemini + RAG + LangGraph**
###### *Elaborado por Soraya Malpica*
---

### **Descripción del Proyecto**

Este notebook implementa un **agente experto en nutrición y entrenamiento deportivo** que combina:

- **RAG (Retrieval-Augmented Generation)**: recupera contexto relevante de una base de conocimiento vectorial antes de responder
- **Google Gemini**: como LLM principal y motor de embeddings
- **ChromaDB**: base de datos vectorial para almacenar y consultar los documentos
- **LangGraph**: framework de agente con memoria de conversación entre turnos

### **Base de Conocimiento**

| Documento | Páginas | Contenido |
|-----------|---------|----------|
| `Doc1_Fundamentos_Nutricion_Deportiva.pdf` | 10 | Macronutrientes, hidratación, nutrición perioperativa |
| `Doc2_Planificacion_Entrenamiento.pdf` | 9 | Periodización, fuerza, resistencia, composición corporal |
| `Doc3_Recuperacion_Suplementacion.pdf` | 10 | Recuperación, suplementos, casos prácticos, lesiones |

### **Arquitectura del Sistema**

```
Usuario → [Pregunta] → LangGraph Agent
                            ↓
                    [Nodo: retrieve]
                    ChromaDB query (top-k chunks)
                            ↓
                    [Nodo: generate]
                    Gemini LLM + System Prompt
                    + Contexto RAG
                    + Historial conversación
                            ↓
                    [Respuesta] → Usuario
```
---

## **PASO 1 - Instalación de Dependencias**

Este proyecto gestiona las dependencias con **`uv`**. Antes de ejecutar el notebook asegúrate de haberlo lanzado desde el entorno correcto:

```bash
# Opción A - uv lanza jupyter dentro del entorno automáticamente (recomendado)
uv run jupyter notebook asistente_deportivo_rag.ipynb

# Opción B - activar el entorno manualmente y luego abrir jupyter
source .venv/bin/activate        # Linux / macOS
.venv\\Scripts\\activate          # Windows
jupyter notebook asistente_deportivo_rag.ipynb
```

Si aún no has instalado las dependencias, ejecuta esto **en la terminal** (no en el notebook):
```bash
uv sync
```

### **PASO 1.A - Carga de Dependencias**

In [3]:
# LLM y Embeddings
# from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
# from langchain.agents import create_agent
# from langchain_core.messages import HumanMessage, BaseMessage

# from langgraph.graph.message import add_messages
# from typing import Annotated, TypedDict

# from chromadb import Chroma # Revisar

# Configuración de API Key y variables del entorno
from dotenv import load_dotenv
import os

load_dotenv()

# Procesamiento de Documentos PDF (Chunking)
import glob
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### **PASO 1.B - Configuración de Constantes**

In [5]:
# Configuración de rutas
PDF_DIR = './docs'

PDF_FILES = [
    'Doc1_Fundamentos_Nutricion_Deportiva.pdf',
    'Doc2_Planificacion_Entrenamiento.pdf',
    'Doc3_Recuperacion_Suplementacion.pdf'
]

# API Key
API_KEY = os.getenv("GEMINI_API_KEY")
MODEL = 'gemini-2.5-flash'
MODEL_2 = 'gemini-2.5-flash-lite'

## **PASO 2 — Carga y Procesamiento de Documentos PDF (Chunking)**
Procesamos los 3 PDFs con:
- **Chunk size**: 800 tokens, suficiente para capturar tablas y párrafos completos
- **Chunk overlap**: 100 tokens, evita perder contexto en los bordes de los chunks
- **Separadores semánticos**: prioriza separar por párrafos, luego por frases

### **PASO 2.A - Carga de PDFs**

In [17]:
documentos = []

for pdf_file in PDF_FILES:
    pdf_path = os.path.join(PDF_DIR, pdf_file)
    
    if not os.path.exists(pdf_path):
        print(f'No encontrado: {pdf_path} - asegúrate de colocar los PDFs en "{PDF_DIR}"')
        continue
    
    carga = PyPDFLoader(pdf_path)
    docs = carga.load()
    
    # Añadir metadatos a cada página
    for doc in docs:
        doc.metadata["source_file"] = pdf_file
        doc.metadata["domain"] = "nutricion_deportiva"
    
    documentos.extend(docs)
    print(f"{pdf_file}: {len(docs)} páginas cargadas")

print(f"\nTotal páginas cargadas: {len(documentos)}")

Doc1_Fundamentos_Nutricion_Deportiva.pdf: 10 páginas cargadas
Doc2_Planificacion_Entrenamiento.pdf: 9 páginas cargadas
Doc3_Recuperacion_Suplementacion.pdf: 10 páginas cargadas

Total páginas cargadas: 29


### **PASO 2.B - Chunking**

In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,           # 800 caracteres por chunk
    chunk_overlap=100,        # Solapamiento para preservar contexto
    length_function=len,
    separators=['\n\n', '\n', '. ', ' ', '']  # Separación semántica
)

chunks = text_splitter.split_documents(documentos)

print(f'-> Total chunks generados: {len(chunks)}')
print(f'-> Tamaño medio de chunk: {sum(len(c.page_content) for c in chunks) // len(chunks)} caracteres')

# Ejemplo de chunk
print('\nEjemplo de chunk')
print(f'Fuente: {chunks[5].metadata.get('source_file', 'N/A')} | Página: {chunks[5].metadata.get('page', 'N/A')}')
print(chunks[5].page_content[:300] + '...')

-> Total chunks generados: 77
-> Tamaño medio de chunk: 626 caracteres

Ejemplo de chunk
Fuente: Doc1_Fundamentos_Nutricion_Deportiva.pdf | Página: 2
CAPÍTULO 2: MACRONUTRIENTES EN EL DEPORTE
2.1 Carbohidratos: El Combustible Principal
Los carbohidratos (CHO) son la fuente de energía preferida por el músculo esquelético durante el ejercicio
de intensidad moderada-alta. Se almacenan en forma de glucógeno en el músculo (aproximadamente
400-500 g) y...


---

In [5]:
# LLM y Embeddings
llm = ChatGoogleGenerativeAI(
    model=MODEL, 
    temperature=0, google_api_key=API_KEY
    )

embeddings = GoogleGenerativeAIEmbeddings(
    model=MODEL, 
    google_api_key=API_KEY)

In [ ]:
# Base de conocimiento vectorial: ChromaDB
vectorstore = Chroma.from_documents(
    documents = documentos, # Generar documentos
    embedding = embeddings,
    collection_name = 'master_ia_doc' # Cambiar
)

In [ ]:
# Framework de agente: LangGraph (Ccon LangChain como base)
@tool
def buscar_documentacion(query: str) -> str:
    '''
    Busca información relevante en la documentación del **master de IA**.
    Úsala cuando necesites información sobre LangChain, RAG, LangGraph, embeddings, **AWS BedRock**
    '''
    docs = retriever.invoke(query)
    if not docs:
        return 'No enocntré información relevante sobre se tema'
    
    resultados = []

    for i, doc in enumerate(docs,1):
        fuente = doc.metadata.get('source', 'desconocida')
        resultados.append(f'[Fuente: {fuente}] \n {doc.page_content}')

    return '\n\n'.join(resultados)

In [ ]:
resultado = buscar_documentacion.invoke({'query': '¿Qué es el RAG'}) # Cambiar

In [ ]:
system_prompt = '''
Eres un asistente experto en **IA Generativa para un máster universitario**.
Tienes acceso a documentación del *máster**^. Cuando el usuario pregunta algo técnico, SIEMPRE usa la herramienta,
`buscar_documentacion` para fundamentar tu respuesta. Si la pregunta es de conversación general, puedes responder
directamente. 
Responde siempre en español
'''

In [ ]:
agente_rag = create_agent(
    model=llm,
    tools=[buscar_documentacion],
    system_prompt=system_prompt
)

In [ ]:
def preguntar_al_agente(pregunta: str):
    '''
    Función helper para hacer preguntas al agente y ver el proceso
    '''
    print('\n{"="*60}')
    print(f'PRegunta: {pregunta}')
    print('='*60)

    respuesta = agente_rag.invoke({
        'messages': [HumanMessage(content=pregunta)]
    })

    # Mostrar el proceso paso a paso
    print('\nProceso del agente:')
    for msg in respuesta['messages']:
        tipo = msg.__class__.__name__
        if tipo == 'HumanMessage':
            print(f'Usuario: {msg.content[:100]}')
        elif tipo == 'AIMessage':
            if msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f'Agente decide usar herramientas: {tc['name']}')
                    print(f' Query: {tc['args'].get('query', '')}')

            else:
                print(f'\n Respuesta final: \n {msg.content}')
        elif tipo == 'ToolMessage':
            print(f'Resultado recuperado: {msg.content[:150]}...')

In [ ]:
# Pregunta 1: Requiere búsqueda
preguntar_al_agente('Que ingredienes tiene la toritll de patata?') # Cambiar